# Exercise 5: Back-translation experiment

*Machine Translation · version 0.1.0*

> Configure a separate Kyrgyz→Kazakh NLLB model (`src_lang="kir_Cyrl"`, `tgt_lang="kaz_Cyrl"`, forced target token `kaz_Cyrl`) to back-translate a small training-only monolingual Kyrgyz corpus (1,000 sentences from Wikipedia or news, disjoint from development/test source documents) into Kazakh, producing synthetic Kazakh. Combine this synthetic data with the available real training pairs and retrain the model. Tune the synthetic-data amount on validation, then compare fixed systems on the same untouched test set. Does adding back-translated data improve chrF++? How sensitive is the gain to the size of the back-translated corpus?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" torch transformers sentencepiece sacrebleu requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`). NLLB-200 distilled 600M is loaded from `NLLB_PATH` (a local copy) or from the Hugging Face Hub at the revision pinned in the book; it runs in float32 on the CPU, about 1.7 s per sentence on four cores. Translations are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes where it stopped. Test data come from the commit-pinned TIL release (X-WMT test sets, MIT licence); the X-WMT files are tokenised, and the code removes the spaces around punctuation before scoring (a heuristic, see `detok`). Scores use SacreBLEU with its default settings: BLEU with the `13a` tokeniser, chrF, and chrF++ (`word_order=2`).

**What runs here.** The data side of the experiment: monolingual Kyrgyz news sentences from the KyrgyzNER training split (24.kg, MIT licence), filtered to 5–30 words, deduplicated and checked against every Kyrgyz side of X-WMT so that no test sentence enters the synthetic data. 60 sentences are back-translated (Kyrgyz→Kazakh) and translated back (Kazakh→Kyrgyz) as a filter signal. The retraining needs a GPU: write the synthetic pairs as `train.tsv` rows (synthetic Kazakh source, real Kyrgyz target) and run `code/ch10-mt/train.py`, which enforces document-grouped, non-overlapping splits.

In [ ]:
import collections, hashlib, os, re, statistics, time
from pathlib import Path
import requests

# Commit-pinned TIL release files (X-WMT test sets and the corpus README); MIT licence.
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation) so references match raw MT output."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)       # Sialkot ’ taki -> Sialkot’taki
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt(src, tgt, n=None):
    """The first n sentence pairs of the X-WMT test set for src-tgt (X-WMT language codes)."""
    s = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{src}").read_text(encoding="utf-8").splitlines()
    t = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{tgt}").read_text(encoding="utf-8").splitlines()
    assert len(s) == len(t), (src, tgt)
    pairs = [(detok(a), detok(b)) for a, b in zip(s, t) if a.strip() and b.strip()]
    return pairs[:n] if n else pairs

NLLB_CODE = {"az": "azj_Latn", "ba": "bak_Cyrl", "kk": "kaz_Cyrl", "ky": "kir_Cyrl", "tr": "tur_Latn",
             "uz": "uzn_Latn", "en": "eng_Latn", "ru": "rus_Cyrl"}
NLLB = {}

def nllb():
    """NLLB-200 distilled 600M: from NLLB_PATH (a local copy) or the pinned Hub revision; float32, CPU."""
    if not NLLB:
        import torch
        from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
        path = os.environ.get("NLLB_PATH", "facebook/nllb-200-distilled-600M")
        rev = None if Path(path).exists() else "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
        NLLB["tok"] = NllbTokenizer.from_pretrained(path, revision=rev)
        NLLB["model"] = AutoModelForSeq2SeqLM.from_pretrained(path, revision=rev, torch_dtype=torch.float32).eval()
        NLLB["torch"] = torch
        torch.set_num_threads(os.cpu_count() or 1)
    return NLLB["tok"], NLLB["model"]

CACHE_FILE = CACHE / "nllb-translations.jsonl"       # translations are cached, so a run can resume
LAST_SECONDS = 0.0

def _cached():
    import json
    out = {}
    if CACHE_FILE.exists():
        for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[tuple(r["key"])] = (r["out"], r["capped"], r["sec"])
    return out

def translate(sents, src, tgt, beams=1, batch=8, max_new=256):
    """Translate with explicit source and target language tokens (greedy unless beams > 1).
    Returns (outputs, number of outputs that reached the generation ceiling without an
    end-of-sequence token). Results are cached per sentence in CACHE_FILE, with the generation
    time of their batch divided by its size; LAST_SECONDS holds the total for the last call."""
    global LAST_SECONDS
    import json
    cache = _cached()
    key = lambda s: [src, tgt, beams, max_new, s]
    todo = [s for s in dict.fromkeys(sents) if tuple(key(s)) not in cache]
    if todo:
        tok, model = nllb()
        tok.src_lang = NLLB_CODE[src]
        CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
        with NLLB["torch"].inference_mode(), CACHE_FILE.open("a", encoding="utf-8") as f:
            for i in range(0, len(todo), batch):
                chunk = todo[i:i + batch]
                t0 = time.time()
                enc = tok(chunk, return_tensors="pt", padding=True)
                gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]),
                                     num_beams=beams, do_sample=False, max_new_tokens=max_new)
                sec = (time.time() - t0) / len(chunk)
                for s, row, text in zip(chunk, gen, tok.batch_decode(gen, skip_special_tokens=True)):
                    capped = int(row[-1] != tok.eos_token_id and row[-1] != tok.pad_token_id)
                    cache[tuple(key(s))] = (text, capped, sec)
                    f.write(json.dumps({"key": key(s), "out": text, "capped": capped, "sec": sec}, ensure_ascii=False) + "\n")
                f.flush()
    res = [cache[tuple(key(s))] for s in sents]
    LAST_SECONDS = sum(r[2] for r in res)
    return [r[0] for r in res], sum(r[1] for r in res)

def metrics(hyps, refs):
    """Corpus BLEU (13a), chrF and chrF++ with SacreBLEU, rounded, plus the chrF++ signature."""
    from sacrebleu.metrics import BLEU, CHRF
    bleu, chrf, chrfpp = BLEU(tokenize="13a"), CHRF(), CHRF(word_order=2)
    r = {"BLEU": bleu.corpus_score(hyps, [refs]).score, "chrF": chrf.corpus_score(hyps, [refs]).score,
         "chrF++": chrfpp.corpus_score(hyps, [refs]).score}
    return {k: round(v, 2) for k, v in r.items()}, str(chrfpp.get_signature())

In [ ]:
# Back-translation, data side only: monolingual Kyrgyz news sentences that are not part of any
# X-WMT test set (the KyrgyzNER training split, 24.kg news), translated Kyrgyz -> Kazakh.
KYRGYZNER = ("https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/"
             "data/KyrgyzNER_TRAIN.jsonl.txt")
sents, cur = [], []
for line in fetch(KYRGYZNER).read_text(encoding="utf-8").splitlines() + [""]:
    if not line.strip():
        if cur: sents.append(detok(" ".join(cur)))
        cur = []
    elif not line.startswith("# "):
        cur.append(line.split("\t")[0])
test_ky = {t for s in ("kk", "uz", "en", "ru", "tr", "az", "ba") for _, t in xwmt(s, "ky")} | \
          {a for t in ("sah", "kaa") for a, _ in xwmt("ky", t)}
mono = [s for s in dict.fromkeys(sents) if 5 <= len(s.split()) <= 30 and s not in test_ky][:60]
print(f"{len(sents)} KyrgyzNER training sentences; {len(mono)} unique sentences of 5-30 words used, none in an X-WMT test set")
synthetic_kk, capped = translate(mono, "ky", "kk")              # synthetic Kazakh sources
roundtrip_ky, _ = translate(synthetic_kk, "kk", "ky")            # back again, as a filter signal
print(f"{capped} back-translations at the length ceiling")
from sacrebleu.metrics import CHRF
rt = [CHRF(word_order=2).sentence_score(b, [a]).score for a, b in zip(mono, roundtrip_ky)]
copied = sum(k.strip() == y.strip() for k, y in zip(synthetic_kk, mono))
print(f"round-trip chrF++ (Kyrgyz -> Kazakh -> Kyrgyz vs. original): median {statistics.median(rt):.1f}, "
      f"min {min(rt):.1f}; {sum(r < 50 for r in rt)} of {len(rt)} below 50; outputs identical to the input: {copied}")
i = min(range(len(rt)), key=lambda j: rt[j])
print("\nlowest round-trip pair:\n  ky :", mono[i], "\n  kk*:", synthetic_kk[i], "\n  ky':", roundtrip_ky[i])

## Worked solution

Output of `experiments/ch10_exercises/run_ch10_exercises.py` (NLLB-200 distilled 600M, CPU, float32):

```
7033 KyrgyzNER training sentences; 60 unique sentences of 5-30 words used, none in an X-WMT test set
0 back-translations at the length ceiling
round-trip chrF++ (Kyrgyz -> Kazakh -> Kyrgyz vs. original): median 58.3, min 26.0; 21 of 60 below 50; outputs identical to the input: 0

lowest round-trip pair:
  ky : Фостердик үй-бүлөлөр айына 6 миң сом маяна жана камкордукка алган ар бир балага дагы жогоркудай өлчөмдөгүчө акча алат. 
  kk*: Foster отбасылары ай сайын 6 мың сом жалақы алады және баласына асырау үшін одан да көп ақша алады. 
  ky': Foster үй-бүлөлөрү ай сайын 6 миң сомдон айлык алышат жана баласын багуу үчүн андан да көп акча алышат.
```

**The synthetic data.** None of the 60 back-translations reached the length ceiling, and none simply copied the Kyrgyz input. Round-trip chrF++ (Kyrgyz → Kazakh → Kyrgyz against the original) has a median of 58.3, but 21 of the 60 sentences score below 50. The lowest pair shows the typical failures: *Фостердик үй-бүлөлөр* 'foster families' becomes *Foster отбасылары* with the English word copied in Latin script, and the end of the sentence ('for each child taken into care, the same amount again') becomes 'more money for raising the child'. A low round-trip score does not prove that the Kazakh side is wrong, since the error can sit in either leg, but it is a cheap way to rank sentences for inspection or filtering.

**The retraining question.** Whether the synthetic data improve chrF++, and how the gain depends on the amount, can only be answered by the training runs, which were not done here. Tune the synthetic amount (for example 0, 1,000, 5,000 and 20,000 sentences, with and without round-trip filtering) on a development set, and compare the fixed systems once on the untouched test set.

## Your turn — extension

1. Back-translate 1,000 sentences, filter by round-trip chrF++, and train with `code/ch10-mt/train.py` on a GPU.
2. Detect Latin-script words in the synthetic Kazakh side and measure how often English words are copied.